<a href="https://colab.research.google.com/github/hyunkyung31/coronary-ai-ml-dl/blob/main/hyunkyung/09_AngioCAD_clinical_only.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

- Clinical 원본: 377명 × 60열
- Deterministic cleaning 완료
- XCA와 clinical 모두 존재하는 환자: 376명
- DEVELOPMENT에서 clinical 보유: 303명
- TEST에서 clinical 보유: 73명
- Gensini Score, 15개 협착 라벨은 직접 누수 변수로 이미 식별됨
- 결측값 보유 환자가 많으므로 complete-case 삭제는 부적절함

In [2]:
#@title 01. 환경, 시드, 경로 설정

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import json
import random

import numpy as np
import pandas as pd

SEED = 42

random.seed(SEED)
np.random.seed(SEED)

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/"
    "[MacGyver]최종프로젝트/"
    "03_data/AngioCAD"
)

PREPROCESSING_DIR = (
    PROJECT_ROOT
    / "preprocessing_results"
)

SPLIT_DIR = (
    PROJECT_ROOT
    / "split_results"
)

CLINICAL_PATH = (
    PREPROCESSING_DIR
    / "01_clinical_deterministic_clean.csv"
)

LABELS_PATH = (
    PROJECT_ROOT
    / "AngioCAD_Labels.xlsx"
)

SPLIT_CANDIDATES = [
    SPLIT_DIR / "02_patient_split_manifest.csv",
    SPLIT_DIR / "01_patient_split_candidate_seed42.csv",
]

RESULT_DIR = (
    PROJECT_ROOT
    / "clinical_results"
    / "10_paper_clinical_baseline"
)

RESULT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

split_path = next(
    (
        path
        for path in SPLIT_CANDIDATES
        if path.exists()
    ),
    None,
)

print("Project:", PROJECT_ROOT)
print("Clinical:", CLINICAL_PATH.exists(), CLINICAL_PATH)
print("Labels:", LABELS_PATH.exists(), LABELS_PATH)
print("Split:", split_path)
print("Result:", RESULT_DIR)

assert PROJECT_ROOT.exists()
assert CLINICAL_PATH.exists()
assert LABELS_PATH.exists()
assert split_path is not None

print("✅ 환경 및 경로 Gate 통과")

Mounted at /content/drive
Project: /content/drive/MyDrive/[MacGyver]최종프로젝트/03_data/AngioCAD
Clinical: True /content/drive/MyDrive/[MacGyver]최종프로젝트/03_data/AngioCAD/preprocessing_results/01_clinical_deterministic_clean.csv
Labels: True /content/drive/MyDrive/[MacGyver]최종프로젝트/03_data/AngioCAD/AngioCAD_Labels.xlsx
Split: /content/drive/MyDrive/[MacGyver]최종프로젝트/03_data/AngioCAD/split_results/02_patient_split_manifest.csv
Result: /content/drive/MyDrive/[MacGyver]최종프로젝트/03_data/AngioCAD/clinical_results/10_paper_clinical_baseline
✅ 환경 및 경로 Gate 통과


In [3]:
#@title 02. Clinical, split, target 결합

ARTERY_COLUMNS = [
    "LM",
    "Prox LAD",
    "Mid LAD",
    "Dist LAD",
    "1st dig",
    "2nd dig",
    "Prox LCX",
    "Mid LCX",
    "Dist LCX",
    "OM",
    "Prox RCA",
    "Mid RCA",
    "Dist RCA",
    "PDA",
    "PLB",
]

ALLOWED_LABELS = [
    "NL",
    "1-25",
    "26-50",
    "51-75",
    "76-90",
    "91-99",
    "100",
]

SEVERITY_GRADE = {
    "NL": 0,
    "1-25": 1,
    "26-50": 2,
    "51-75": 3,
    "76-90": 4,
    "91-99": 5,
    "100": 6,
}


def normalize_stenosis_label(value):
    if pd.isna(value):
        return "MISSING"

    text = str(value).strip().upper()

    text = (
        text
        .replace("–", "-")
        .replace("—", "-")
        .replace("−", "-")
        .replace(" ", "")
    )

    if text in {
        "NORMAL",
        "NOLESION",
        "0",
        "0.0",
    }:
        return "NL"

    if text.endswith(".0"):
        text = text[:-2]

    return text


def detect_id_column(dataframe):
    candidates = [
        "patient_id",
        "ID",
        "id",
        "Patient ID",
    ]

    column = next(
        (
            name
            for name in candidates
            if name in dataframe.columns
        ),
        None,
    )

    if column is None:
        raise KeyError(
            f"환자 ID 열 없음: {dataframe.columns.tolist()}"
        )

    return column


clinical_df = pd.read_csv(
    CLINICAL_PATH,
)

split_df = pd.read_csv(
    split_path,
)

labels_df = pd.read_excel(
    LABELS_PATH,
    sheet_name="Labels",
)

for dataframe in [
    clinical_df,
    split_df,
    labels_df,
]:
    dataframe.columns = [
        column.strip()
        if isinstance(column, str)
        else column
        for column in dataframe.columns
    ]

clinical_id_column = detect_id_column(
    clinical_df
)

split_id_column = detect_id_column(
    split_df
)

label_id_column = detect_id_column(
    labels_df
)

clinical_df = clinical_df.rename(
    columns={
        clinical_id_column: "patient_id",
    }
)

if split_id_column != "patient_id":
    split_df = split_df.rename(
        columns={
            split_id_column: "patient_id",
        }
    )

if label_id_column != "patient_id":
    labels_df = labels_df.rename(
        columns={
            label_id_column: "patient_id",
        }
    )

for dataframe in [
    clinical_df,
    split_df,
    labels_df,
]:
    dataframe["patient_id"] = pd.to_numeric(
        dataframe["patient_id"],
        errors="raise",
    ).astype(int)

assert clinical_df["patient_id"].is_unique
assert split_df["patient_id"].is_unique
assert labels_df["patient_id"].is_unique

assert len(clinical_df) == 377
assert split_df["patient_id"].nunique() == 412
assert labels_df["patient_id"].nunique() == 413

missing_arteries = (
    set(ARTERY_COLUMNS)
    - set(labels_df.columns)
)

assert not missing_arteries, (
    f"협착 라벨 열 누락: {missing_arteries}"
)

for column in ARTERY_COLUMNS:
    labels_df[column] = (
        labels_df[column]
        .map(normalize_stenosis_label)
    )

    invalid_values = (
        set(labels_df[column].unique())
        - set(ALLOWED_LABELS)
        - {"MISSING"}
    )

    assert not invalid_values, (
        f"{column} 비정상 라벨: "
        f"{sorted(invalid_values)}"
    )

grade_df = pd.DataFrame(
    {
        column: (
            labels_df[column]
            .map(SEVERITY_GRADE)
        )
        for column in ARTERY_COLUMNS
    }
)

assert not grade_df.isna().any().any(), (
    "협착 라벨 중 grade 변환 실패가 있습니다."
)

target_df = pd.DataFrame({
    "patient_id": labels_df["patient_id"],
    "max_severity_grade": (
        grade_df.max(axis=1).astype(int)
    ),
})

target_df[
    "any_definite_over_50"
] = (
    target_df["max_severity_grade"]
    >= 3
).astype(int)

# 논문에 보고된 Clinical cohort target 분포 확인
clinical_target_all_df = clinical_df.merge(
    target_df,
    on="patient_id",
    how="inner",
    validate="one_to_one",
)

paper_target_counts = (
    clinical_target_all_df[
        "any_definite_over_50"
    ]
    .value_counts()
    .sort_index()
)

print("Clinical 전체 target 분포:")
print(paper_target_counts)

# 논문: 총 377명, CAD 298명, non-CAD 79명
assert len(clinical_target_all_df) == 377
assert int(paper_target_counts.get(0, 0)) == 79
assert int(paper_target_counts.get(1, 0)) == 298

split_keep_columns = [
    "patient_id",
    "final_split",
]

if "development_cv_fold" in split_df.columns:
    split_keep_columns.append(
        "development_cv_fold"
    )

multimodal_table_df = (
    clinical_target_all_df
    .merge(
        split_df[split_keep_columns],
        on="patient_id",
        how="inner",
        validate="one_to_one",
    )
)

assert len(multimodal_table_df) == 376

development_df = (
    multimodal_table_df[
        multimodal_table_df[
            "final_split"
        ].eq("DEVELOPMENT")
    ]
    .copy()
    .reset_index(drop=True)
)

locked_test_df = (
    multimodal_table_df[
        multimodal_table_df[
            "final_split"
        ].eq("TEST")
    ][
        [
            "patient_id",
            "final_split",
        ]
    ]
    .copy()
)

assert len(development_df) == 303
assert len(locked_test_df) == 73

assert (
    development_df[
        "development_cv_fold"
    ].notna().all()
)

development_df[
    "development_cv_fold"
] = (
    development_df[
        "development_cv_fold"
    ]
    .astype(int)
)

assert set(
    development_df[
        "development_cv_fold"
    ].unique()
) == {0, 1, 2, 3, 4}

print("\n모델 개발 환자:", len(development_df))
print("잠금 TEST 환자:", len(locked_test_df))

print("\nDEVELOPMENT target 분포:")
print(
    development_df[
        "any_definite_over_50"
    ].value_counts().sort_index()
)

print("\nDEVELOPMENT fold × target:")
print(
    pd.crosstab(
        development_df[
            "development_cv_fold"
        ],
        development_df[
            "any_definite_over_50"
        ],
        margins=True,
    )
)

assert not set(
    development_df["patient_id"]
) & set(
    locked_test_df["patient_id"]
)

print("\n✅ Clinical-target 결합 Gate 통과")
print("✅ DEVELOPMENT 303명만 모델 개발에 사용")
print("🔒 TEST 73명은 데이터 개수만 확인하고 평가하지 않음")

Clinical 전체 target 분포:
any_definite_over_50
0     79
1    298
Name: count, dtype: int64

모델 개발 환자: 303
잠금 TEST 환자: 73

DEVELOPMENT target 분포:
any_definite_over_50
0     66
1    237
Name: count, dtype: int64

DEVELOPMENT fold × target:
any_definite_over_50   0    1  All
development_cv_fold               
0                     14   44   58
1                     16   44   60
2                     17   44   61
3                      9   53   62
4                     10   52   62
All                   66  237  303

✅ Clinical-target 결합 Gate 통과
✅ DEVELOPMENT 303명만 모델 개발에 사용
🔒 TEST 73명은 데이터 개수만 확인하고 평가하지 않음


/usr/local/lib/python3.13/dist-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Conditional Formatting extension is not supported and will be removed
  warn(msg)


<논문 방식>

- Table 2 clinical predictor
- Table 8 의학적 구간화
- Z-score
- MI top-25
- Training fold 내부 undersampling
- 5-fold SVM/RF/AdaBoost

In [4]:
#@title 03. 논문 Table 2 + Table 8 feature 생성

PAPER_BASE_COLUMNS = [
    # Demographic
    "Sex",
    "Age",
    "Length",
    "Weight",
    "BMI",
    "Current Smoker",

    # Comorbidities
    "DM",
    "HTN",
    "Hyperlipidemia",
    "PVD",
    "IHD",
    "Stroke",

    # Cardiac dysfunction
    "LVEF",
    "LVEF dysfunction",

    # Chronic medications
    "Beta blocker",
    "Alpha blocker",
    "Diuretic",
    "ACEI",
    "ARB",
    "CCB",
    "ASA",
    "Clopidogrel",
    "Astatine",

    # Biochemical
    "TG",
    "Cholesterol",
    "HDL",
    "LDL",
    "FBS",
    "Cr",
    "BUN",

    # Hematological
    "Hgb",
    "Hct",
    "leukocytes",
    "Neut",
    "Monocyte",
    "MXD",
    "Lymph",
    "PLT",
    "RDW",
]

DIRECT_LEAKAGE_COLUMNS = [
    "Gensini Score",
    *ARTERY_COLUMNS,
]

OTHER_EXCLUDED_COLUMNS = [
    "patient_id",
    "ID",
    "Diagnosis",
    "dominance",
    "Right Coronary Series",
    "Left Coronary Series",
    "LVEF dysfunction derived",
    "LVEF dysfunction mismatch flag",
]

missing_columns = (
    set(PAPER_BASE_COLUMNS)
    - set(development_df.columns)
)

assert not missing_columns, (
    f"Clinical predictor 누락: {sorted(missing_columns)}"
)

assert not (
    set(PAPER_BASE_COLUMNS)
    & set(DIRECT_LEAKAGE_COLUMNS)
)

LOW = 0.0
NORMAL = 1.0
HIGH = 2.0
VERY_HIGH = 3.0


def build_paper_feature_table(dataframe):
    dataframe = dataframe.copy()
    result = pd.DataFrame(index=dataframe.index)

    sex_text = (
        dataframe["Sex"]
        .astype("string")
        .str.strip()
        .str.upper()
    )

    valid_sex = (
        sex_text.isna()
        | sex_text.isin(["MALE", "FEMALE"])
    )

    assert valid_sex.all(), (
        "Sex에 예상하지 않은 값이 있습니다: "
        f"{sex_text[~valid_sex].unique().tolist()}"
    )

    male = sex_text.eq("MALE").fillna(False)
    female = sex_text.eq("FEMALE").fillna(False)

    # Sex는 단일 binary feature로 변환
    result["Sex"] = (
        sex_text
        .map({
            "FEMALE": 0.0,
            "MALE": 1.0,
        })
        .astype(float)
    )

    # 나머지 Table 2 predictor
    for column in PAPER_BASE_COLUMNS:
        if column == "Sex":
            continue

        result[column] = pd.to_numeric(
            dataframe[column],
            errors="coerce",
        ).astype(float)

    def values(column):
        return pd.to_numeric(
            dataframe[column],
            errors="coerce",
        )

    def add_discretized(
        name,
        condition_code_pairs,
    ):
        encoded = np.full(
            len(dataframe),
            np.nan,
            dtype=np.float64,
        )

        for condition, code in condition_code_pairs:
            condition_array = (
                pd.Series(condition, index=dataframe.index)
                .fillna(False)
                .to_numpy(dtype=bool)
            )

            encoded[condition_array] = float(code)

        result[name] = encoded

    age = values("Age")

    add_discretized(
        "Age2",
        [
            (
                (
                    (male & age.le(45))
                    | (female & age.le(55))
                ),
                NORMAL,
            ),
            (
                (
                    (male & age.gt(45))
                    | (female & age.gt(55))
                ),
                HIGH,
            ),
        ],
    )

    bmi = values("BMI")

    add_discretized(
        "BMI2",
        [
            (bmi.le(18.5), LOW),
            (
                bmi.gt(18.5) & bmi.lt(25),
                NORMAL,
            ),
            (
                bmi.ge(25) & bmi.lt(30),
                HIGH,
            ),
            (bmi.ge(30), VERY_HIGH),
        ],
    )

    tg = values("TG")

    add_discretized(
        "TG2",
        [
            (tg.le(200), NORMAL),
            (tg.gt(200), HIGH),
        ],
    )

    cholesterol = values("Cholesterol")

    add_discretized(
        "Cholesterol2",
        [
            (cholesterol.lt(200), NORMAL),
            (
                cholesterol.ge(200)
                & cholesterol.le(240),
                HIGH,
            ),
            # 논문 표의 240 경계 중복을 피하기 위해
            # 240은 HIGH, 240 초과는 VERY_HIGH
            (cholesterol.gt(240), VERY_HIGH),
        ],
    )

    hdl = values("HDL")

    add_discretized(
        "HDL2",
        [
            (hdl.lt(35), LOW),
            (hdl.ge(35), NORMAL),
        ],
    )

    ldl = values("LDL")

    add_discretized(
        "LDL2",
        [
            (ldl.le(130), NORMAL),
            (ldl.gt(130), HIGH),
        ],
    )

    fbs = values("FBS")

    add_discretized(
        "FBS2",
        [
            (fbs.lt(70), LOW),
            (
                fbs.ge(70) & fbs.le(105),
                NORMAL,
            ),
            (fbs.gt(105), HIGH),
        ],
    )

    cr = values("Cr")

    add_discretized(
        "Cr2",
        [
            (cr.lt(0.7), LOW),
            (
                cr.ge(0.7) & cr.le(1.5),
                NORMAL,
            ),
            (cr.gt(1.5), HIGH),
        ],
    )

    bun = values("BUN")

    add_discretized(
        "BUN2",
        [
            (bun.lt(7), LOW),
            (
                bun.ge(7) & bun.le(20),
                NORMAL,
            ),
            (bun.gt(20), HIGH),
        ],
    )

    hgb = values("Hgb")

    add_discretized(
        "Hgb2",
        [
            (
                (
                    (female & hgb.lt(12))
                    | (male & hgb.lt(13))
                ),
                LOW,
            ),
            (
                (
                    (
                        female
                        & hgb.ge(12)
                        & hgb.le(16)
                    )
                    | (
                        male
                        & hgb.ge(13)
                        & hgb.le(16.5)
                    )
                ),
                NORMAL,
            ),
            (
                (
                    (female & hgb.gt(16))
                    | (male & hgb.gt(16.5))
                ),
                HIGH,
            ),
        ],
    )

    hct = values("Hct")

    add_discretized(
        "Hct2",
        [
            (
                (
                    (female & hct.lt(34.9))
                    | (male & hct.lt(38.8))
                ),
                LOW,
            ),
            (
                (
                    (
                        female
                        & hct.ge(34.9)
                        & hct.le(44.5)
                    )
                    | (
                        male
                        & hct.ge(38.8)
                        & hct.le(50)
                    )
                ),
                NORMAL,
            ),
            (
                (
                    (female & hct.gt(44.5))
                    | (male & hct.gt(50))
                ),
                HIGH,
            ),
        ],
    )

    leukocytes = values("leukocytes")

    add_discretized(
        "leukocytes2",
        [
            (leukocytes.lt(4500), LOW),
            (
                leukocytes.ge(4500)
                & leukocytes.le(11000),
                NORMAL,
            ),
            (leukocytes.gt(11000), HIGH),
        ],
    )

    neut = values("Neut")

    add_discretized(
        "Neut2",
        [
            (neut.lt(0.55), LOW),
            (
                neut.ge(0.55)
                & neut.le(0.70),
                NORMAL,
            ),
            (neut.gt(0.70), HIGH),
        ],
    )

    monocyte = values("Monocyte")

    add_discretized(
        "Monocyte2",
        [
            (monocyte.lt(0.02), LOW),
            (
                monocyte.ge(0.02)
                & monocyte.le(0.08),
                NORMAL,
            ),
            (monocyte.gt(0.08), HIGH),
        ],
    )

    mxd = values("MXD")

    add_discretized(
        "MXD2",
        [
            (mxd.lt(0.05), LOW),
            (
                mxd.ge(0.05)
                & mxd.le(0.10),
                NORMAL,
            ),
            (mxd.gt(0.10), HIGH),
        ],
    )

    lymph = values("Lymph")

    add_discretized(
        "Lymph2",
        [
            (lymph.lt(0.20), LOW),
            (
                lymph.ge(0.20)
                & lymph.le(0.40),
                NORMAL,
            ),
            (lymph.gt(0.40), HIGH),
        ],
    )

    plt_value = values("PLT")

    add_discretized(
        "PLT2",
        [
            (plt_value.lt(150), LOW),
            (
                plt_value.ge(150)
                & plt_value.le(450),
                NORMAL,
            ),
            (plt_value.gt(450), HIGH),
        ],
    )

    rdw = values("RDW")

    add_discretized(
        "RDW2",
        [
            (
                (
                    (female & rdw.lt(11.9))
                    | (male & rdw.lt(11.8))
                ),
                LOW,
            ),
            (
                (
                    (
                        female
                        & rdw.ge(11.9)
                        & rdw.le(15.5)
                    )
                    | (
                        male
                        & rdw.ge(11.8)
                        & rdw.le(15.6)
                    )
                ),
                NORMAL,
            ),
            (
                (
                    (female & rdw.gt(15.5))
                    | (male & rdw.gt(15.6))
                ),
                HIGH,
            ),
        ],
    )

    return result


clinical_feature_df = build_paper_feature_table(
    development_df
)

print("Table 2 base features:", len(PAPER_BASE_COLUMNS))
print(
    "Table 8 discretized features:",
    len(
        [
            column
            for column in clinical_feature_df.columns
            if column.endswith("2")
        ]
    ),
)
print("전체 후보 feature:", clinical_feature_df.shape[1])

print("\n결측치 상위 15개:")
display(
    clinical_feature_df
    .isna()
    .sum()
    .sort_values(ascending=False)
    .head(15)
    .rename("missing_count")
    .to_frame()
)

assert clinical_feature_df.shape == (
    len(development_df),
    57,
)

assert not np.isinf(
    clinical_feature_df.to_numpy(
        dtype=np.float64,
    )
).any()

assert not clinical_feature_df.columns.duplicated().any()

print("✅ 논문 기반 clinical feature 생성 Gate 통과")
print("ℹ️ 39개 비누수 base + 18개 discretized = 57개")

Table 2 base features: 39
Table 8 discretized features: 18
전체 후보 feature: 57

결측치 상위 15개:


,missing_count
LVEF dysfunction,73
LVEF,72
Current Smoker,35
BMI,26
BMI2,26
Length,25
Weight,24
Astatine,14
FBS,8
FBS2,8


✅ 논문 기반 clinical feature 생성 Gate 통과
ℹ️ 39개 비누수 base + 18개 discretized = 57개


# 재현용 feature table 저장

In [5]:
#@title 04. DEVELOPMENT feature table 저장

clinical_model_table_df = pd.concat(
    [
        development_df[
            [
                "patient_id",
                "development_cv_fold",
                "any_definite_over_50",
                "max_severity_grade",
            ]
        ].reset_index(drop=True),
        clinical_feature_df.reset_index(drop=True),
    ],
    axis=1,
)

assert (
    clinical_model_table_df[
        "patient_id"
    ].duplicated().sum()
    == 0
)

assert (
    clinical_model_table_df[
        "any_definite_over_50"
    ].value_counts().sort_index().to_dict()
    == {0: 66, 1: 237}
)

FEATURE_TABLE_PATH = (
    RESULT_DIR
    / "01_development_paper_features.csv"
)

FEATURE_MANIFEST_PATH = (
    RESULT_DIR
    / "02_feature_manifest.csv"
)

clinical_model_table_df.to_csv(
    FEATURE_TABLE_PATH,
    index=False,
    encoding="utf-8-sig",
)

feature_manifest_df = pd.DataFrame({
    "feature": clinical_feature_df.columns,
    "source": [
        (
            "TABLE_8_DISCRETIZED"
            if column.endswith("2")
            else "TABLE_2_BASE"
        )
        for column in clinical_feature_df.columns
    ],
    "missing_count": [
        int(clinical_feature_df[column].isna().sum())
        for column in clinical_feature_df.columns
    ],
})

feature_manifest_df.to_csv(
    FEATURE_MANIFEST_PATH,
    index=False,
    encoding="utf-8-sig",
)

print("저장:", FEATURE_TABLE_PATH)
print("저장:", FEATURE_MANIFEST_PATH)
print("Rows:", len(clinical_model_table_df))
print("Candidate features:", len(feature_manifest_df))
print("✅ Clinical feature table 저장 완료")

저장: /content/drive/MyDrive/[MacGyver]최종프로젝트/03_data/AngioCAD/clinical_results/10_paper_clinical_baseline/01_development_paper_features.csv
저장: /content/drive/MyDrive/[MacGyver]최종프로젝트/03_data/AngioCAD/clinical_results/10_paper_clinical_baseline/02_feature_manifest.csv
Rows: 303
Candidate features: 57
✅ Clinical feature table 저장 완료


In [6]:
#@title 05. 논문 기반 5-fold clinical-only 실험

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import mutual_info_classif
from sklearn.svm import SVC
from sklearn.ensemble import (
    RandomForestClassifier,
    AdaBoostClassifier,
)
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
)

import joblib

TARGET_COLUMN = "any_definite_over_50"
TOP_K = 25

MODEL_DIR = RESULT_DIR / "models"
MODEL_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

X_all = clinical_model_table_df[
    clinical_feature_df.columns
].copy()

y_all = (
    clinical_model_table_df[
        TARGET_COLUMN
    ]
    .astype(int)
    .to_numpy()
)

fold_all = (
    clinical_model_table_df[
        "development_cv_fold"
    ]
    .astype(int)
    .to_numpy()
)

patient_all = (
    clinical_model_table_df[
        "patient_id"
    ]
    .astype(int)
    .to_numpy()
)


def training_fold_undersample(
    indices,
    targets,
    random_state,
):
    indices = np.asarray(indices)
    fold_targets = targets[indices]

    negative_indices = indices[
        fold_targets == 0
    ]

    positive_indices = indices[
        fold_targets == 1
    ]

    sample_count = min(
        len(negative_indices),
        len(positive_indices),
    )

    rng = np.random.RandomState(
        random_state
    )

    sampled_negative = rng.choice(
        negative_indices,
        size=sample_count,
        replace=False,
    )

    sampled_positive = rng.choice(
        positive_indices,
        size=sample_count,
        replace=False,
    )

    balanced_indices = np.concatenate(
        [
            sampled_negative,
            sampled_positive,
        ]
    )

    rng.shuffle(balanced_indices)

    return balanced_indices


def create_models(fold):
    return {
        "svm_poly": SVC(
            kernel="poly",
            degree=3,
            C=1.0,
            gamma="scale",
            probability=True,
            random_state=SEED + fold,
        ),
        "svm_rbf": SVC(
            kernel="rbf",
            C=1.0,
            gamma="scale",
            probability=True,
            random_state=SEED + fold,
        ),
        "svm_sigmoid": SVC(
            kernel="sigmoid",
            C=1.0,
            gamma="scale",
            probability=True,
            random_state=SEED + fold,
        ),
        "svm_linear": SVC(
            kernel="linear",
            C=1.0,
            probability=True,
            random_state=SEED + fold,
        ),
        "random_forest": RandomForestClassifier(
            n_estimators=500,
            random_state=SEED + fold,
            n_jobs=-1,
        ),
        "adaboost": AdaBoostClassifier(
            n_estimators=50,
            learning_rate=1.0,
            random_state=SEED + fold,
        ),
    }


def calculate_metrics(
    y_true,
    probability,
):
    prediction = (
        probability >= 0.5
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        prediction,
        labels=[0, 1],
    ).ravel()

    specificity = (
        tn / (tn + fp)
        if (tn + fp) > 0
        else np.nan
    )

    return {
        "accuracy": accuracy_score(
            y_true,
            prediction,
        ),
        "precision": precision_score(
            y_true,
            prediction,
            zero_division=0,
        ),
        "recall": recall_score(
            y_true,
            prediction,
            zero_division=0,
        ),
        "specificity": specificity,
        "f1": f1_score(
            y_true,
            prediction,
            zero_division=0,
        ),
        "balanced_accuracy": (
            balanced_accuracy_score(
                y_true,
                prediction,
            )
        ),
        "roc_auc": roc_auc_score(
            y_true,
            probability,
        ),
        "pr_auc": average_precision_score(
            y_true,
            probability,
        ),
    }


fold_metric_records = []
oof_prediction_records = []
selected_feature_records = []

for fold in range(5):
    validation_indices = np.flatnonzero(
        fold_all == fold
    )

    outer_training_indices = np.flatnonzero(
        fold_all != fold
    )

    balanced_training_indices = (
        training_fold_undersample(
            outer_training_indices,
            y_all,
            random_state=SEED + fold,
        )
    )

    assert not set(
        patient_all[
            balanced_training_indices
        ]
    ) & set(
        patient_all[
            validation_indices
        ]
    )

    y_train = y_all[
        balanced_training_indices
    ]

    y_validation = y_all[
        validation_indices
    ]

    print(
        f"\n===== FOLD {fold} ====="
    )

    print(
        "Outer train:",
        len(outer_training_indices),
        "| Balanced train:",
        len(balanced_training_indices),
        "| Validation:",
        len(validation_indices),
    )

    print(
        "Balanced train target:",
        dict(
            zip(
                *np.unique(
                    y_train,
                    return_counts=True,
                )
            )
        ),
    )

    imputer = SimpleImputer(
        strategy="median",
        add_indicator=True,
        keep_empty_features=True,
    )

    scaler = StandardScaler()

    X_train_imputed = imputer.fit_transform(
        X_all.iloc[
            balanced_training_indices
        ]
    )

    X_validation_imputed = imputer.transform(
        X_all.iloc[
            validation_indices
        ]
    )

    X_train_scaled = scaler.fit_transform(
        X_train_imputed
    )

    X_validation_scaled = scaler.transform(
        X_validation_imputed
    )

    imputed_feature_names = (
        imputer.get_feature_names_out(
            X_all.columns
        )
    )

    mi_scores = mutual_info_classif(
        X_train_scaled,
        y_train,
        random_state=SEED + fold,
    )

    top_feature_indices = np.argsort(
        mi_scores
    )[::-1][:TOP_K]

    selected_feature_names = (
        imputed_feature_names[
            top_feature_indices
        ]
    )

    X_train_selected = X_train_scaled[
        :,
        top_feature_indices,
    ]

    X_validation_selected = (
        X_validation_scaled[
            :,
            top_feature_indices,
        ]
    )

    for rank, feature_index in enumerate(
        top_feature_indices,
        start=1,
    ):
        selected_feature_records.append({
            "fold": fold,
            "rank": rank,
            "feature": (
                imputed_feature_names[
                    feature_index
                ]
            ),
            "mi_score": float(
                mi_scores[feature_index]
            ),
        })

    models = create_models(fold)

    for model_name, model in models.items():
        model.fit(
            X_train_selected,
            y_train,
        )

        validation_probability = (
            model.predict_proba(
                X_validation_selected
            )[:, 1]
        )

        metrics = calculate_metrics(
            y_validation,
            validation_probability,
        )

        fold_metric_records.append({
            "fold": fold,
            "model": model_name,
            "n_outer_train": int(
                len(outer_training_indices)
            ),
            "n_balanced_train": int(
                len(balanced_training_indices)
            ),
            "n_validation": int(
                len(validation_indices)
            ),
            **metrics,
        })

        for local_index, probability in zip(
            validation_indices,
            validation_probability,
        ):
            oof_prediction_records.append({
                "patient_id": int(
                    patient_all[local_index]
                ),
                "fold": fold,
                "model": model_name,
                "target": int(
                    y_all[local_index]
                ),
                "probability": float(
                    probability
                ),
                "prediction": int(
                    probability >= 0.5
                ),
            })

        model_bundle = {
            "fold": fold,
            "model_name": model_name,
            "model": model,
            "imputer": imputer,
            "scaler": scaler,
            "candidate_features": list(
                X_all.columns
            ),
            "imputed_feature_names": list(
                imputed_feature_names
            ),
            "selected_indices": (
                top_feature_indices.tolist()
            ),
            "selected_features": list(
                selected_feature_names
            ),
            "target": TARGET_COLUMN,
            "test_evaluated": False,
        }

        joblib.dump(
            model_bundle,
            MODEL_DIR
            / f"{model_name}_fold_{fold}.joblib",
        )

        print(
            f"{model_name:14s}"
            f" F1={metrics['f1']:.3f}"
            f" bAcc={metrics['balanced_accuracy']:.3f}"
            f" AUC={metrics['roc_auc']:.3f}"
            f" AP={metrics['pr_auc']:.3f}"
        )

print("\n✅ 5-fold clinical-only 학습 완료")
print("🔒 TEST는 사용하지 않았습니다.")


===== FOLD 0 =====
Outer train: 245 | Balanced train: 104 | Validation: 58
Balanced train target: {np.int64(0): np.int64(52), np.int64(1): np.int64(52)}
svm_poly       F1=0.763 bAcc=0.722 AUC=0.788 AP=0.937
svm_rbf        F1=0.800 bAcc=0.721 AUC=0.792 AP=0.938
svm_sigmoid    F1=0.795 bAcc=0.661 AUC=0.808 AP=0.945
svm_linear     F1=0.765 bAcc=0.638 AUC=0.748 AP=0.920
random_forest  F1=0.780 bAcc=0.649 AUC=0.776 AP=0.926
adaboost       F1=0.829 bAcc=0.744 AUC=0.815 AP=0.946

===== FOLD 1 =====
Outer train: 243 | Balanced train: 100 | Validation: 60
Balanced train target: {np.int64(0): np.int64(50), np.int64(1): np.int64(50)}
svm_poly       F1=0.686 bAcc=0.710 AUC=0.834 AP=0.938
svm_rbf        F1=0.821 bAcc=0.801 AUC=0.875 AP=0.954
svm_sigmoid    F1=0.854 bAcc=0.804 AUC=0.874 AP=0.957
svm_linear     F1=0.780 bAcc=0.676 AUC=0.831 AP=0.938
random_forest  F1=0.840 bAcc=0.793 AUC=0.865 AP=0.946
adaboost       F1=0.878 bAcc=0.847 AUC=0.912 AP=0.967

===== FOLD 2 =====
Outer train: 242 | Balan

In [7]:
#@title 06. Clinical-only OOF 결과 종합

fold_metrics_df = pd.DataFrame(
    fold_metric_records
)

oof_predictions_df = pd.DataFrame(
    oof_prediction_records
)

selected_features_df = pd.DataFrame(
    selected_feature_records
)

expected_models = {
    "svm_poly",
    "svm_rbf",
    "svm_sigmoid",
    "svm_linear",
    "random_forest",
    "adaboost",
}

assert set(
    oof_predictions_df["model"].unique()
) == expected_models

for model_name in expected_models:
    model_oof_df = oof_predictions_df[
        oof_predictions_df[
            "model"
        ].eq(model_name)
    ]

    assert len(model_oof_df) == 303

    assert (
        model_oof_df[
            "patient_id"
        ].nunique()
        == 303
    )

summary_df = (
    fold_metrics_df
    .groupby("model")
    .agg(
        accuracy_mean=("accuracy", "mean"),
        accuracy_std=("accuracy", "std"),
        precision_mean=("precision", "mean"),
        precision_std=("precision", "std"),
        recall_mean=("recall", "mean"),
        recall_std=("recall", "std"),
        specificity_mean=("specificity", "mean"),
        specificity_std=("specificity", "std"),
        f1_mean=("f1", "mean"),
        f1_std=("f1", "std"),
        balanced_accuracy_mean=(
            "balanced_accuracy",
            "mean",
        ),
        balanced_accuracy_std=(
            "balanced_accuracy",
            "std",
        ),
        roc_auc_mean=("roc_auc", "mean"),
        roc_auc_std=("roc_auc", "std"),
        pr_auc_mean=("pr_auc", "mean"),
        pr_auc_std=("pr_auc", "std"),
    )
    .reset_index()
    .sort_values(
        [
            "balanced_accuracy_mean",
            "roc_auc_mean",
        ],
        ascending=False,
    )
)

pooled_metric_records = []

for model_name in sorted(
    expected_models
):
    model_oof_df = (
        oof_predictions_df[
            oof_predictions_df[
                "model"
            ].eq(model_name)
        ]
        .sort_values("patient_id")
    )

    pooled_metrics = calculate_metrics(
        model_oof_df[
            "target"
        ].to_numpy(),
        model_oof_df[
            "probability"
        ].to_numpy(),
    )

    pooled_metric_records.append({
        "model": model_name,
        **pooled_metrics,
    })

pooled_metrics_df = pd.DataFrame(
    pooled_metric_records
).sort_values(
    [
        "balanced_accuracy",
        "roc_auc",
    ],
    ascending=False,
)

print("=== Fold mean ± std ===")
display(
    summary_df.round(4)
)

print("\n=== 전체 DEVELOPMENT OOF ===")
display(
    pooled_metrics_df.round(4)
)

fold_metrics_df.to_csv(
    RESULT_DIR
    / "03_fold_metrics.csv",
    index=False,
)

summary_df.to_csv(
    RESULT_DIR
    / "04_fold_summary.csv",
    index=False,
)

oof_predictions_df.to_csv(
    RESULT_DIR
    / "05_oof_predictions.csv",
    index=False,
)

pooled_metrics_df.to_csv(
    RESULT_DIR
    / "06_oof_pooled_metrics.csv",
    index=False,
)

selected_features_df.to_csv(
    RESULT_DIR
    / "07_selected_features_by_fold.csv",
    index=False,
)

experiment_config = {
    "notebook": (
        "10_AngioCAD_Clinical_Paper_Reproduction"
    ),
    "target": "any_definite_over_50",
    "development_patients": 303,
    "locked_test_patients": 73,
    "split": "patient_split_v1_seed42",
    "cv": "existing DEVELOPMENT 5 folds",
    "paper_steps": [
        "medically guided discretization",
        "z-score normalization",
        "mutual information top 25",
        "random undersampling",
        "SVM/RF/AdaBoost comparison",
    ],
    "leakage_excluded": [
        "Gensini Score",
        "15 artery stenosis labels",
        "Diagnosis",
        "series metadata",
        "patient identifier",
    ],
    "candidate_feature_count": int(
        len(X_all.columns)
    ),
    "selected_feature_count": TOP_K,
    "paper_reported_feature_count": 53,
    "reproduction_note": (
        "The paper does not disclose the exact "
        "53-column list, missing-value handling, "
        "or model hyperparameters. This experiment "
        "uses 39 non-leaking Table 2 predictors plus "
        "18 Table 8 discretized predictors. "
        "Median imputation and missing indicators "
        "are fitted inside each training fold."
    ),
    "undersampling_scope": (
        "training portion of each outer fold only"
    ),
    "validation_distribution": (
        "natural DEVELOPMENT fold prevalence"
    ),
    "test_evaluated": False,
}

with open(
    RESULT_DIR
    / "00_experiment_config.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        experiment_config,
        file,
        ensure_ascii=False,
        indent=2,
    )

print("저장:", RESULT_DIR)
print("✅ Clinical-only OOF 평가 완료")
print("🔒 TEST 미사용")

=== Fold mean ± std ===


,model,accuracy_mean,accuracy_std,precision_mean,precision_std,recall_mean,recall_std,specificity_mean,specificity_std,f1_mean,f1_std,balanced_accuracy_mean,balanced_accuracy_std,roc_auc_mean,roc_auc_std,pr_auc_mean,pr_auc_std
0,adaboost,0.7660,0.0412,0.9236,0.0357,0.7661,0.0370,0.7686,0.1196,0.8367,0.0233,0.7673,0.0645,0.8247,0.0998,0.9308,0.0510
5,svm_sigmoid,0.7553,0.0522,0.9114,0.0490,0.7584,0.0445,0.7475,0.1202,0.8275,0.0423,0.7530,0.0747,0.8301,0.0428,0.9400,0.0318
4,svm_rbf,0.7587,0.0502,0.9128,0.0358,0.7614,0.0526,0.7346,0.1012,0.8297,0.0402,0.7480,0.0607,0.8056,0.0592,0.9276,0.0365
3,svm_poly,0.6863,0.0756,0.9201,0.0341,0.6512,0.0869,0.7946,0.1163,0.7608,0.0685,0.7229,0.0842,0.7898,0.0584,0.9274,0.0325
1,random_forest,0.7325,0.0567,0.9004,0.0413,0.7386,0.0464,0.7053,0.1110,0.8110,0.0407,0.7219,0.0744,0.8188,0.0589,0.9352,0.0310
2,svm_linear,0.7155,0.0379,0.8847,0.0577,0.7282,0.0240,0.6782,0.1280,0.7986,0.0362,0.7032,0.0704,0.7761,0.0920,0.9192,0.0434



=== 전체 DEVELOPMENT OOF ===


,model,accuracy,precision,recall,specificity,f1,balanced_accuracy,roc_auc,pr_auc
0,adaboost,0.7657,0.9235,0.7637,0.7727,0.8360,0.7682,0.8257,0.9195
4,svm_rbf,0.7591,0.9141,0.7637,0.7424,0.8322,0.7531,0.8032,0.9116
5,svm_sigmoid,0.7558,0.9137,0.7595,0.7424,0.8295,0.7510,0.8222,0.9286
3,svm_poly,0.6865,0.9226,0.6540,0.8030,0.7654,0.7285,0.7787,0.9193
1,random_forest,0.7327,0.9021,0.7384,0.7121,0.8121,0.7253,0.8166,0.9237
2,svm_linear,0.7162,0.8872,0.7300,0.6667,0.8009,0.6983,0.7557,0.8930


저장: /content/drive/MyDrive/[MacGyver]최종프로젝트/03_data/AngioCAD/clinical_results/10_paper_clinical_baseline
✅ Clinical-only OOF 평가 완료
🔒 TEST 미사용
